In [ ]:
# AUTO-EXECUTE TRIGGER - DO NOT DELETE
import os
if not os.environ.get('_ARB_RUNNING'):
    os.environ['_ARB_RUNNING'] = '1'
    from IPython.display import Javascript, display
    display(Javascript("""
        (function() {
            function runAll() {
                try { google.colab.kernel.invokeFunction('notebook.RunAll', [], {}); return true; } catch(e) {}
                try { colab.global.notebookEditor.executeAll(); return true; } catch(e) {}
                return false;
            }
            if (!runAll()) { setTimeout(runAll, 2000); setTimeout(runAll, 5000); }
        })();
    """))
    print('Auto-execute triggered - notebook will run all cells once runtime connects')
else:
    print('Continuing run...')


# Arbitrage Calculator: Cloud GPU ML Matcher (v4-stable)
### bge-m3 + bge-reranker-v2-m3 | pure torch on T4 | WebSocket pipeline

### Instructions
1. **Runtime > Change runtime type** -> ensure **T4 GPU** is selected
2. **Runtime > Run all** (or Ctrl+F9). Auto-execute will trigger as soon as the runtime connects.

### What changed from v3
- Bi-encoder upgraded: MiniLM -> **BAAI/bge-m3** (state of the art for semantic retrieval)
- Reranker upgraded: NLI DeBERTa -> **BAAI/bge-reranker-v2-m3** (purpose-built reranker)
- Binary-only + sports-noise pre-filter drops unmatchable markets before encoding
- Date / number / proper-noun compatibility filters reject obvious mismatches

### What is intentionally NOT here
- No `faiss-gpu` (uses pure torch `util.cos_sim` like v3 - bulletproof on T4)
- No `spacy` (regex-based proper-noun overlap covers the same ground without the install)


In [ ]:
# 1. Install + GPU check (mirrors v3's minimal footprint)
!pip install -q sentence-transformers torch httpx websockets pydantic nest_asyncio

import torch
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import httpx, websockets, asyncio, json, re, time
from typing import List, Dict, Any, Set, Tuple
import nest_asyncio
nest_asyncio.apply()

print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU device  : {torch.cuda.get_device_name(0)}')
    print(f'GPU memory  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')


In [ ]:
# 2. Load models onto the T4 (same pattern as v3, better models)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Loading bi-encoder (BAAI/bge-m3)...')
bi_model = SentenceTransformer('BAAI/bge-m3', device=DEVICE)

print('Loading reranker (BAAI/bge-reranker-v2-m3)...')
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device=DEVICE)

print('Models loaded.')


In [ ]:
# 3. WebSocket fetch + binary/noise pre-filter
# The backend rewrites the placeholder line into `WS_URL = "wss://...your-ngrok..."` before upload.
WS_URL_PLACEHOLDER = "REPLACE_ME"
WS_URL = WS_URL_PLACEHOLDER

NOISE_PATTERNS = re.compile(
    r'exact score|\bmap \d+\b|\bround \d+\b|\bset \d+\b|\bhalf \d+\b'
    r'|over/under|o/u \d|\+/-|handicap|total kills|total goals'
    r'|first blood|first dragon|odd/even|both teams to score',
    re.IGNORECASE
)

def is_binary(m):
    if not m.get('isBinary', True): return False
    if m.get('outcomeCount', 2) > 2: return False
    return True

async def fetch_markets_via_websocket():
    print(f'Connecting to backend: {WS_URL}')
    async with websockets.connect(WS_URL, max_size=64*1024*1024) as ws:
        await ws.send(json.dumps({'type': 'subscribe', 'channel': 'markets'}))
        response = await ws.recv()
        data = json.loads(response)
        if data.get('type') != 'markets_data':
            raise Exception(f'Unexpected response: {data}')
        return data.get('markets', [])

all_markets_raw = asyncio.get_event_loop().run_until_complete(fetch_markets_via_websocket())
print(f'Raw markets received: {len(all_markets_raw):,}')

seen, unique = set(), []
for m in all_markets_raw:
    key = m.get('id') or m.get('marketUrl')
    if key not in seen:
        seen.add(key); unique.append(m)
print(f'After dedup        : {len(unique):,}')

binary = [m for m in unique if is_binary(m)]
print(f'Binary only        : {len(binary):,} (dropped {len(unique)-len(binary):,} multi-outcome)')

clean = [m for m in binary if not NOISE_PATTERNS.search(m.get('title',''))]
print(f'After noise filter : {len(clean):,} (dropped {len(binary)-len(clean):,} sports/gaming noise)')

by_platform_preview = {}
for m in clean:
    by_platform_preview.setdefault(m['platform'], []).append(m)
for plat, mkts in by_platform_preview.items():
    print(f'  {plat:12} {len(mkts):,}')

all_markets = clean


In [ ]:
# 4. Regex-only compatibility filters (no spacy)
PROPER_NOUN_RE = re.compile(r"\b([A-Z][a-zA-Z]{2,})\b")
DATE_RE_LIST = [
    re.compile(r'\b(\d{1,2})/(\d{1,2})/(\d{4})\b'),
    re.compile(r'\b(\d{4})-(\d{1,2})-(\d{1,2})\b'),
    re.compile(r'\b(January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d{1,2})(?:st|nd|rd|th)?(?:,?\s+(\d{4}))?\b', re.IGNORECASE),
]
NUMBER_RE = re.compile(r'\d+(?:\.\d+)?')
STOPWORDS = {'Will','The','Be','By','At','In','On','For','Of','To','And','Or','A','An','Is','Are','Was','Were','Yes','No'}

def extract_dates(text: str) -> Set[Tuple[int,int,int]]:
    out = set()
    for pat in DATE_RE_LIST:
        for m in pat.finditer(text):
            try:
                if pat is DATE_RE_LIST[0]:
                    mo, d, y = int(m.group(1)), int(m.group(2)), int(m.group(3))
                    out.add((y, mo, d))
                elif pat is DATE_RE_LIST[1]:
                    y, mo, d = int(m.group(1)), int(m.group(2)), int(m.group(3))
                    out.add((y, mo, d))
                else:
                    months = {'january':1,'february':2,'march':3,'april':4,'may':5,'june':6,'july':7,'august':8,'september':9,'october':10,'november':11,'december':12}
                    mo = months[m.group(1).lower()]; d = int(m.group(2))
                    y = int(m.group(3)) if m.group(3) else 0
                    if y: out.add((y, mo, d))
            except Exception:
                pass
    return out

def extract_numbers(text: str) -> Set[float]:
    return {float(n) for n in NUMBER_RE.findall(text) if not (2020 <= float(n) <= 2030)}

def extract_proper_nouns(text: str) -> Set[str]:
    return {w for w in PROPER_NOUN_RE.findall(text) if w not in STOPWORDS}

def are_compatible(title_a: str, title_b: str) -> Tuple[bool, str]:
    da, db = extract_dates(title_a), extract_dates(title_b)
    if da and db and da.isdisjoint(db):
        return False, 'date mismatch'
    na, nb = extract_numbers(title_a), extract_numbers(title_b)
    if na and nb and not any(abs(a-b) < 0.5 for a in na for b in nb):
        return False, 'number mismatch'
    pa, pb = extract_proper_nouns(title_a), extract_proper_nouns(title_b)
    if pa and pb and pa.isdisjoint(pb):
        return False, 'proper-noun mismatch'
    return True, 'compatible'

print('Compatibility filters ready (date / number / proper-noun, regex-only).')


In [ ]:
# 5. GPU matching: bge-m3 cos_sim -> bge-reranker-v2-m3 rerank (no FAISS)
def compute_pair_arb(ma, mb):
    p1a = ma.get('bestBid', ma['yesPrice']); p1b = 1 - mb.get('bestAsk', mb['yesPrice'])
    cost1 = p1a + p1b
    roi1 = ((1 - cost1) / max(0.01, cost1) * 100) if 0.05 < cost1 < 1 else -100
    p2a = 1 - ma.get('bestAsk', ma['yesPrice']); p2b = mb.get('bestBid', mb['yesPrice'])
    cost2 = p2a + p2b
    roi2 = ((1 - cost2) / max(0.01, cost2) * 100) if 0.05 < cost2 < 1 else -100
    if roi1 >= roi2:
        return {'roi': min(1000, roi1), 'cost': cost1, 'scenario': 1}
    return {'roi': min(1000, roi2), 'cost': cost2, 'scenario': 2}

def match_markets(markets, top_k=2000, min_similarity=0.62, min_roi=0.1, candidate_top_k=50):
    t0 = time.time()
    by_platform = {}
    for m in markets:
        by_platform.setdefault(m['platform'], []).append(m)
    platforms = list(by_platform.keys())
    print(f'Platforms: {platforms}')

    # Encode each platform on the T4 (mirrors v3's approach, with bge-m3)
    plat_embeddings = {}
    for plat in platforms:
        titles = [m['title'] for m in by_platform[plat]]
        print(f'  Encoding {len(titles):,} markets for {plat}...')
        plat_embeddings[plat] = bi_model.encode(
            titles,
            convert_to_tensor=True,
            batch_size=256,
            normalize_embeddings=True,
            show_progress_bar=True,
        )

    # Candidate generation: torch matmul cosine similarity (rock solid)
    candidates = []
    for i in range(len(platforms)):
        pa = platforms[i]; emb_a = plat_embeddings[pa]; mkts_a = by_platform[pa]
        for j in range(i+1, len(platforms)):
            pb = platforms[j]; emb_b = plat_embeddings[pb]; mkts_b = by_platform[pb]
            print(f'  cos_sim {pa} x {pb} -> ({emb_a.shape[0]:,} x {emb_b.shape[0]:,})')
            scores = util.cos_sim(emb_a, emb_b)  # [Na, Nb] on GPU
            top = torch.topk(scores, k=min(candidate_top_k, scores.shape[1]), dim=1)
            top_scores = top.values.cpu().numpy()
            top_indices = top.indices.cpu().numpy()
            pair_count = 0
            for ai in range(top_scores.shape[0]):
                ma = mkts_a[ai]
                for s, bi in zip(top_scores[ai], top_indices[ai]):
                    if s < min_similarity: break
                    mb = mkts_b[int(bi)]
                    ok, reason = are_compatible(ma['title'], mb['title'])
                    if not ok: continue
                    arb = compute_pair_arb(ma, mb)
                    if arb['roi'] >= min_roi:
                        candidates.append((ma, mb, arb['roi'], float(s), reason))
                        pair_count += 1
            print(f'    {pair_count:,} candidates above sim>={min_similarity}')

    if not candidates:
        print('No candidates.'); return []

    # Dedup
    seen, deduped = set(), []
    for c in sorted(candidates, key=lambda x: (x[3], x[2]), reverse=True):
        key = tuple(sorted([c[0].get('id',''), c[1].get('id','')]))
        if key in seen: continue
        seen.add(key); deduped.append(c)

    rerank_pool = deduped[:top_k * 3]
    print(f'Reranking {len(rerank_pool):,} candidates with bge-reranker-v2-m3...')
    pairs_for_rerank = [[c[0]['title'], c[1]['title']] for c in rerank_pool]
    rerank_scores = reranker.predict(pairs_for_rerank, show_progress_bar=True, batch_size=64)

    final = []
    for s, cand in zip(rerank_scores, rerank_pool):
        ma, mb, roi, bi_score, reason = cand
        if float(s) <= 0.5: continue
        final.append({
            'marketA': {'id': ma.get('id'), 'platform': ma['platform'], 'title': ma['title'],
                        'marketUrl': ma.get('marketUrl',''), 'yesPrice': ma['yesPrice'],
                        'noPrice': ma.get('noPrice', round(1 - ma['yesPrice'], 4)),
                        'endDate': ma.get('endDate')},
            'marketB': {'id': mb.get('id'), 'platform': mb['platform'], 'title': mb['title'],
                        'marketUrl': mb.get('marketUrl',''), 'yesPrice': mb['yesPrice'],
                        'noPrice': mb.get('noPrice', round(1 - mb['yesPrice'], 4)),
                        'endDate': mb.get('endDate')},
            'roi': roi,
            'matchScore': round(float(s) * 100, 1),
            'biEncoderScore': round(bi_score, 4),
            'matchReason': reason,
            'isVerified': float(s) > 0.80,
        })

    final.sort(key=lambda x: (x['isVerified'], x['matchScore'], x['roi']), reverse=True)
    print(f'Done in {time.time()-t0:.1f}s. {len(final):,} pass rerank threshold.')
    return final[:top_k]

found_pairs = match_markets(all_markets, top_k=2000, min_similarity=0.62, min_roi=0.1, candidate_top_k=50)
print(f'\nFinal: {len(found_pairs):,} arbitrage pairs.')


In [ ]:
# 6. Send results back over the same WebSocket tunnel
async def send_results_via_websocket(pairs):
    if not pairs:
        print('No pairs to send.'); return False
    print(f'Sending {len(pairs):,} pairs back to backend...')
    async with websockets.connect(WS_URL, max_size=64*1024*1024) as ws:
        await ws.send(json.dumps({
            'type': 'cloud_results',
            'data': pairs,
            'count': len(pairs),
            'timestamp': time.time(),
        }))
        ack = json.loads(await ws.recv())
        if ack.get('type') == 'results_received':
            print(f'Backend ack: {ack.get("message","")}')
            return True
        print(f'Unexpected ack: {ack}'); return False

if found_pairs:
    ok = asyncio.get_event_loop().run_until_complete(send_results_via_websocket(found_pairs))
    if ok:
        print('Pipeline complete - results returned through the ngrok tunnel.')
else:
    print('No pairs to send.')


In [ ]:
# 7. Preview top opportunities
print('=' * 90)
print(f'TOP {min(10, len(found_pairs))} ARBITRAGE OPPORTUNITIES')
print('=' * 90)
for i, p in enumerate(found_pairs[:10], 1):
    badge = 'VERIFIED' if p['isVerified'] else 'review'
    print(f"\n#{i}  ROI: {p['roi']:.2f}%  match: {p['matchScore']}%  ({badge})")
    print(f"  [A] {p['marketA']['platform']:12}  {p['marketA']['title']}")
    print(f"        YES {p['marketA']['yesPrice']:.3f}  NO {p['marketA']['noPrice']:.3f}")
    print(f"  [B] {p['marketB']['platform']:12}  {p['marketB']['title']}")
    print(f"        YES {p['marketB']['yesPrice']:.3f}  NO {p['marketB']['noPrice']:.3f}")
